In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 画像パスを指定（analysis.ipynbからの相対パス）
IMAGE_PATH = "../images/sample.jpg"  # ← 画像ファイルを入れる場所

# 画像読み込み
img_bgr = cv2.imread(IMAGE_PATH)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# 表示
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(img_gray, cmap='gray')
axes[1].set_title("Grayscale")
plt.tight_layout()
plt.show()

print(f"画像サイズ: {img_gray.shape}")  # (高さ, 幅)

In [ ]:
# スケールバーを除去する前に、縮尺を取得する

In [ ]:
# 画像の高さ・幅を取得
h, w = img_gray.shape
print(f"高さ: {h}px, 幅: {w}px")

# スケールバーが写っている下部領域をマスク（除外）
# 画像下部10%をカット（後で調整可能）
crop_bottom = int(h * 0.90)
img_cropped = img_gray[:crop_bottom, :]

# 確認表示
plt.figure(figsize=(10, 6))
plt.imshow(img_cropped, cmap='gray')
plt.title(f"Cropped (Scalebar removed) → {img_cropped.shape}")
# plt.axhline(y=crop_bottom-10, color='red', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Non-Maximum Suppression (NMS) - 近すぎるblobを統合する
def remove_duplicates(keypoints, min_distance=50):
    """中心間距離がmin_distance以下のblobは近い方を削除"""
    if len(keypoints) == 0:
        return keypoints
    
    pts = np.array([kp.pt for kp in keypoints])
    sizes = np.array([kp.size for kp in keypoints])
    keep = []
    suppressed = set()
    
    # 大きいblobを優先して残す
    order = np.argsort(-sizes)
    
    for i in order:
        if i in suppressed:
            continue
        keep.append(i)
        for j in order:
            if i == j or j in suppressed:
                continue
            dist = np.linalg.norm(pts[i] - pts[j])
            if dist < min_distance:
                suppressed.add(j)
    
    return [keypoints[i] for i in keep]

# ブロブ（塊）検出のパラメータ設定
params = cv2.SimpleBlobDetector_Params()

# 面積によるフィルタリング（ピクセル単位の針のサイズに合わせて調整すること）
# 一旦調整済み、（縮尺に応じて変更する必要あり）
# params.filterByArea = True
# params.minArea = 5000
# params.maxArea = 40000

params.filterByArea = True
params.minArea = 11000    # 小さすぎるblobを除外（上げると検出数が減る）
params.maxArea = 40000

# 円形度によるフィルタリング（円ではないため、制限をオフ(False)にしておく）
params.filterByCircularity = False

# 凸性（へこみがないか）によるフィルタリング
# 形がいびつで「内側に大きく凹んでいるような図形」を弾くフィルター
# 50%以上の凸性（ある程度まとまった形をしていること）を条件とする
params.filterByConvexity = True
params.minConvexity = 0.5

# 慣性モーメント（細長さ）によるフィルタリング
params.filterByInertia = False

# 色を反転させた画像で検出（デフォルトの黒ではなく、白を塊として認識させる）
img_inv = cv2.bitwise_not(img_cropped)

detector = cv2.SimpleBlobDetector_create(params)
keypoints = detector.detect(img_inv)

# ダブり除去（NMS）
keypoints_filtered = remove_duplicates(keypoints, min_distance=90)


# 検出されたブロブ（塊）を描画する
img_result = cv2.cvtColor(img_cropped, cv2.COLOR_GRAY2BGR)
for i, kp in enumerate(keypoints_filtered):
    x, y = int(kp.pt[0]), int(kp.pt[1])
    r = int(kp.size / 2)
    cv2.circle(img_result, (x, y), r, (0, 255, 0), 2)
    cv2.putText(img_result, str(i+1), (x-10, y-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB))
plt.title(f"Blob detection: {len(keypoints_filtered)} blobs found")
plt.show()
print(f"Before NMS: {len(keypoints)}, After NMS: {len(keypoints_filtered)}")

In [ ]:
# defective = [4, 8, 12]  # 欠損針の番号（1-indexed）
# normal = [1, 2, 5]      # 比較用の正常針

# fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# for col, idx in enumerate(normal):
#     kp = keypoints_filtered[idx - 1]
#     x, y = int(kp.pt[0]), int(kp.pt[1])
#     r = int(kp.size / 2) + 100  # 少し余白を持って切り出す
#     crop = img_cropped[max(0,y-r):y+r, max(0,x-r):x+r]
#     axes[0, col].imshow(crop, cmap='gray')
#     axes[0, col].set_title(f"Normal #{idx}")
#     axes[0, col].axis('off')

# for col, idx in enumerate(defective):
#     kp = keypoints_filtered[idx - 1]
#     x, y = int(kp.pt[0]), int(kp.pt[1])
#     r = int(kp.size / 2) + 100
#     crop = img_cropped[max(0,y-r):y+r, max(0,x-r):x+r]
#     axes[1, col].imshow(crop, cmap='gray')
#     axes[1, col].set_title(f"Defective #{idx}")
#     axes[1, col].axis('off')

# plt.suptitle("Normal (top) vs Defective (bottom)", fontsize=14)
# plt.tight_layout()
# plt.show()

In [ ]:
# 切り出し元画像と検出済みキーポイント（針）を受け取る関数
def measure_taper_x(img_cropped, keypoints_filtered):
    results = []
    
    # 各針ごとでループ処理
    for i, kp in enumerate(keypoints_filtered):
        x, y = int(kp.pt[0]), int(kp.pt[1]) # 中心座標を取得
        r = int(kp.size / 2) + 100  # 切り出す半径
        
        # 切り出し範囲を画像内に収める
        x1, y1 = max(0, x-r), max(0, y-r)
        x2, y2 = min(img_cropped.shape[1], x+r), min(img_cropped.shape[0], y+r)
        crop = img_cropped[y1:y2, x1:x2]
        
        # 二値化して輪郭を抽出
        blur = cv2.GaussianBlur(crop, (5, 5), 0)    # 5×5のガウシアンぼかしでノイズを減らす
        _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)    # 白黒の二値画像に変換
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # 二値画像から輪郭を検出
        

        # 輪郭が無いときの処理
        if not contours:
            results.append({'id': i+1, 'taper_ratio': None})
            continue
            # Noneを登録して次の針へスキップ。
        
        # 最大輪郭を座標配列にする
        cnt = max(contours, key=cv2.contourArea)
        pts = cnt.reshape(-1, 2)
        
        # 先端＝X座標が最も小さい点
        left_x = pts[:, 0].min()    # 輪郭中で最もxが小さい点＝先端とみなす(他の方向に対応できるように改良必要)
        total_width = pts[:, 0].max() - left_x  # 先端から末端までのx方向の全長
        
        # 指定x位置での縦幅を測る内部関数
        def height_at_x(pts, x_threshold):
            band = pts[pts[:, 0] < x_threshold]
            if len(band) < 2:
                return 0
            return band[:, 1].max() - band[:, 1].min()
        
        # 先端から10%と30%の位置での幅（Y方向の広がり）
        h10 = height_at_x(pts, left_x + total_width * 0.10)
        h30 = height_at_x(pts, left_x + total_width * 0.30)
        
        # テーパー比の計算
        taper_ratio = h10 / h30 if h30 > 0 else None
        
        # 結果を記録
        results.append({
            'id': i+1,
            'taper_ratio': taper_ratio,
            'h10': h10,
            'h30': h30
        })
    
    return results

# 関数の呼び出しと表示
results = measure_taper_x(img_cropped, keypoints_filtered)

print(f"{'#':>3} | {'Taper Ratio':>12} | {'H10':>6} | {'H30':>6}")
print("-" * 35)
for r in results:
    ratio = f"{r['taper_ratio']:>12.3f}" if r.get('taper_ratio') is not None else "        None"
    h10 = f"{r['h10']:>6.1f}" if r.get('h10') is not None else "  None"
    h30 = f"{r['h30']:>6.1f}" if r.get('h30') is not None else "  None"
    print(f"{r['id']:>3} | {ratio} | {h10} | {h30}")

In [ ]:
# グリッドの準備
fig, axes = plt.subplots(3, 5, figsize=(18, 12))    # より多い針の数に対応させる必要あり
axes = axes.ravel()

# 各針ごとでループ処理
for i, kp in enumerate(keypoints_filtered):
    x, y = int(kp.pt[0]), int(kp.pt[1]) # 中心座標を取得
    r = int(kp.size / 2) + 90   # 切り出す半径
    
    # 切り出し範囲を画像内に収める
    x1, y1 = max(0, x-r), max(0, y-r)
    x2, y2 = min(img_cropped.shape[1], x+r), min(img_cropped.shape[0], y+r)
    crop = img_cropped[y1:y2, x1:x2]
    
    # 二値化して輪郭を抽出
    blur = cv2.GaussianBlur(crop, (5, 5), 0)  # 5×5のガウシアンぼかしでノイズを減らす
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)    # 白黒の二値画像に変換
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # 二値画像から輪郭を検出
    
    # 輪郭を描画
    vis = cv2.cvtColor(crop, cv2.COLOR_GRAY2BGR)    # グレースケールの切り出し画像をカラーに変換
    # 輪郭が1つ以上見つかった場合
    if contours:
        # 面積が最大の輪郭を選ぶ
        cnt = max(contours, key=cv2.contourArea)
        # 描画
        cv2.drawContours(vis, [cnt], -1, (0, 255, 0), 2)
    
    # サブプロットに表示
    axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"#{i+1}")
    axes[i].axis('off')

# 余った枠を消す
for j in range(len(keypoints_filtered), len(axes)):
    axes[j].axis('off')

# 表示
plt.suptitle("Contours per needle", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 閾値の設定（テーパー比）
THRESHOLD = 0.5

# 判定

defective_detected = []
normal_detected = []

# 分類ループ
for r in results:
    if r['taper_ratio'] is None:
        continue
    if r['taper_ratio'] >= THRESHOLD:
        defective_detected.append(r['id'])  # 比がしきい値以上なら欠損リストに番号を追加
    else:
        normal_detected.append(r['id']) # それ未満なら正常リストに追加

# 元画像に結果をオーバーレイ
img_final = cv2.cvtColor(img_cropped, cv2.COLOR_GRAY2BGR)

# 描画ループ
for r in results:
    if r['taper_ratio'] is None:
        continue
    kp = keypoints_filtered[r['id'] - 1]
    x, y = int(kp.pt[0]), int(kp.pt[1])
    rad = int(kp.size / 2)
    
    # 色とラベルの振り分け
    if r['taper_ratio'] >= THRESHOLD:
        color = (0, 0, 255)   # 赤 = 欠損
        label = f"#{r['id']} DEF"
    else:
        color = (0, 255, 0)   # 緑 = 正常
        label = f"#{r['id']} OK"
    
    # 円とラベルの描画
    cv2.circle(img_final, (x, y), rad, color, 2)
    cv2.putText(img_final, label, (x - 30, y - rad - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

# 結果画像の表示
plt.figure(figsize=(14, 9))
plt.imshow(cv2.cvtColor(img_final, cv2.COLOR_BGR2RGB))
plt.title(f"Result: {len(defective_detected)} defective / {len(keypoints_filtered)} detected")
plt.axis('off')
plt.show()

print(f"Defective needles: {len(defective_detected)} -> needle #{defective_detected}")
print(f"Normal needles:    {len(normal_detected)}")

In [ ]:
# ============================================================
# 各針の「高さ」と「底辺の長さ」を測定する関数
#
# 【考え方】
#   針の「底辺」は画像上に1本の線として写らない → 根元の輪郭は当てにできない。
#   そこで、先端から伸びる2本の「斜面」を直線とみなして延長し、底辺の位置で交わる2点の距離を「底辺の長さ」とする。
#
# 【slope_frac】
#   先端から高さの何割までを「きれいな斜面」として直線フィットに使うか。
#   根元に近いほど輪郭が乱れるため、先端寄りの部分だけを使う。
#   0.6 = 先端から60%の範囲を斜面とみなす。
# ============================================================

def measure_with_slopes(img_cropped, keypoints_filtered, slope_frac=0.6):
    results = []
    vis_data = []   # 可視化用のデータ（先端・底辺の座標など）を保存する

    # 各針ごとにループ
    for i, kp in enumerate(keypoints_filtered):
        x, y = int(kp.pt[0]), int(kp.pt[1])   # 針の中心座標
        r = int(kp.size / 2) + 100            # 針を切り出す半径（+100は余白）

        # 切り出し範囲が画像の外に出ないよう内側に収める
        x1, y1 = max(0, x-r), max(0, y-r)
        x2, y2 = min(img_cropped.shape[1], x+r), min(img_cropped.shape[0], y+r)
        crop = img_cropped[y1:y2, x1:x2]

        # 二値化して輪郭を抽出（Cell 6と同じ手法）
        blur = cv2.GaussianBlur(crop, (5, 5), 0)   # ノイズ減らし
        _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # 輪郭が無ければNoneを記録して次の針へ
        if not contours:
            results.append({'id': i+1, 'height': None, 'base': None})
            vis_data.append(None)
            continue

        # 最大の輪郭を針の輪郭とみなし、座標配列にする
        cnt = max(contours, key=cv2.contourArea)
        pts = cnt.reshape(-1, 2).astype(np.float64)

        # --- 先端と「軸」の決定 ---
        # 先端＝X座標が最も小さい点（針は左向きなので左端が先端）
        tip = pts[pts[:, 0].argmin()]

        # 重心（輪郭の中心）を求める。cv2.momentsは図形の重心などを計算する関数
        M = cv2.moments(cnt)
        centroid = np.array([M['m10']/M['m00'], M['m01']/M['m00']])

        # 軸＝先端から重心へ向かう方向。これを針の「中心線の向き」とする
        axis = centroid - tip
        axis = axis / np.linalg.norm(axis)   # 長さ1の単位ベクトルに正規化

        # 軸に垂直な方向のベクトル（底辺の向き）
        perp = np.array([-axis[1], axis[0]])

        # --- 各輪郭点を「軸方向」と「軸に垂直な方向」に分解（射影）する ---
        # 「@」はベクトルの内積。射影＝その方向にどれだけ進んだかの距離
        rel = pts - tip                # 先端を原点とした各点の相対座標
        proj_along = rel @ axis        # 軸方向の距離（先端からどれだけ奥か）
        proj_perp  = rel @ perp        # 軸に垂直な距離（中心線からどれだけ横か）

        # 高さ＝軸方向の最大距離（先端から一番奥＝根元まで）
        height = proj_along.max()
        end_pt = tip + axis * height   # 高さの終点（根元側）の座標

        # --- 斜面2本を直線フィットする ---
        # 先端寄り（slope_fracまで）の点だけを使う（根元の乱れを避ける）
        mask = proj_along < height * slope_frac

        # 中心線より上側(perp>0)と下側(perp<0)に点を振り分ける＝上下2本の斜面
        upper = pts[mask & (proj_perp > 0)]
        lower = pts[mask & (proj_perp < 0)]

        # 点群に直線を当てはめる関数。cv2.fitLineは最も当てはまる直線を返す
        def fit_line(line_pts):
            # 戻り値：直線の向き(vx,vy)と、直線が通る点(x0,y0)
            vx, vy, x0, y0 = cv2.fitLine(line_pts.astype(np.float32),
                                          cv2.DIST_L2, 0, 0.01, 0.01).flatten()
            return np.array([x0, y0]), np.array([vx, vy])

        # 2本の直線の交点を求める関数（連立方程式を解く）
        def intersect(p1, d1, p2, d2):
            # 直線1: p1 + t*d1、直線2: p2 + s*d2　の交点を計算
            A = np.array([[d1[0], -d2[0]], [d1[1], -d2[1]]])
            b = p2 - p1
            t, s = np.linalg.solve(A, b)
            return p1 + t * d1

        base = None
        base_pts = None
        # 上下それぞれ2点以上ないと直線が引けないのでチェック
        if len(upper) >= 2 and len(lower) >= 2:
            pu, du = fit_line(upper)   # 上斜面の直線
            pl, dl = fit_line(lower)   # 下斜面の直線

            # 底辺ライン＝高さの終点(end_pt)を通り、perp方向に伸びる直線
            # 上斜面・下斜面をこの底辺ラインまで延長し、交わる点を求める
            top_hit = intersect(pu, du, end_pt, perp)   # 上斜面と底辺ラインの交点
            bot_hit = intersect(pl, dl, end_pt, perp)   # 下斜面と底辺ラインの交点

            # 底辺の長さ＝2交点間の距離
            base = np.linalg.norm(top_hit - bot_hit)
            base_pts = (top_hit, bot_hit)

        # 結果と可視化用データを記録
        results.append({'id': i+1, 'height': height, 'base': base})
        vis_data.append({
            'crop': crop, 'tip': tip, 'end_pt': end_pt,
            'cnt': cnt, 'base_pts': base_pts
        })

    return results, vis_data


# 関数を実行（slope_frac=0.6で測定）
raw, vis_data = measure_with_slopes(img_cropped, keypoints_filtered, slope_frac=0.6)

# ============================================================
# 測定結果の可視化：各針に高さ（青）と底辺（赤）を描く
# ============================================================
fig, axes = plt.subplots(3, 5, figsize=(18, 12))   # 3行5列のグリッド
axes = axes.ravel()

for i, vd in enumerate(vis_data):
    if vd is None:            # 輪郭が取れなかった針は空欄に
        axes[i].axis('off')
        continue

    # 切り出し画像をカラーに変換して輪郭を緑で描画
    vis = cv2.cvtColor(vd['crop'], cv2.COLOR_GRAY2BGR)
    cv2.drawContours(vis, [vd['cnt']], -1, (0, 255, 0), 1)

    tip = vd['tip'].astype(int)
    end_pt = vd['end_pt'].astype(int)

    # 高さ＝先端(赤丸)から終点までの青い線
    cv2.line(vis, tuple(tip), tuple(end_pt), (255, 0, 0), 2)
    cv2.circle(vis, tuple(tip), 6, (0, 0, 255), -1)

    # 底辺＝斜面延長の交点2つを結ぶ赤い線
    if vd['base_pts'] is not None:
        b1 = vd['base_pts'][0].astype(int)
        b2 = vd['base_pts'][1].astype(int)
        cv2.line(vis, tuple(b1), tuple(b2), (0, 0, 255), 2)
        # 先端から底辺への斜面の延長線もオレンジで薄く描画
        cv2.line(vis, tuple(tip), tuple(b1), (255, 150, 0), 1)
        cv2.line(vis, tuple(tip), tuple(b2), (255, 150, 0), 1)

    axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"#{i+1}")
    axes[i].axis('off')

# 余ったグリッド枠を消す
for j in range(len(vis_data), len(axes)):
    axes[j].axis('off')

plt.suptitle("Blue = Height, Red = Base (from slope intersection)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 高さ・底辺を「無次元化」して一覧表示する
#
# 【無次元化とは】
#   ピクセル数そのものは画像の縮尺で変わってしまう。
#   そこで「一番高い針＝1.0」「一番太い針＝1.0」を基準として各針をその比率で表す。
# ============================================================

# 高さ・底辺の両方が取得できた針だけを対象にする
valid = [r for r in raw if r['height'] is not None and r['base'] is not None]

# 基準となる最大値を求める
max_height = max(r['height'] for r in valid)   # 一番高い針の高さ
max_base   = max(r['base']   for r in valid)   # 一番太い針の底辺

# 表のヘッダーを表示
print(f"{'#':>3} | {'Height(px)':>10} | {'Base(px)':>9} | {'H(norm)':>8} | {'B(norm)':>8}")
print("-" * 52)

# 各針の測定値を表示
for r in raw:
    # 測定できなかった針はNoneと表示
    if r['height'] is None or r['base'] is None:
        print(f"{r['id']:>3} |       None |      None |     None |     None")
        continue
    hn = r['height'] / max_height   # 高さの無次元値（0〜1）
    bn = r['base'] / max_base       # 底辺の無次元値（0〜1）
    # px値と無次元値を並べて表示
    print(f"{r['id']:>3} | {r['height']:>10.1f} | {r['base']:>9.1f} | {hn:>8.3f} | {bn:>8.3f}")

# どの針が基準（=1.0）になったかを表示
print()
print(f"Max height (=1.0): #{[r['id'] for r in valid if r['height']==max_height][0]}  ({max_height:.1f}px)")
print(f"Max base   (=1.0): #{[r['id'] for r in valid if r['base']==max_base][0]}  ({max_base:.1f}px)")